# House Price Prediction

**OASIS INFOBYTE — Data Analytics Internship**  
**Track:** Data Analytics  
**Task:** House Price Prediction  
**Candidate:** P J Renu

## Project Objective
Build a machine-learning regression solution that predicts house sale prices from property characteristics. The workflow covers data loading, data quality checks, exploratory analysis, preprocessing, model comparison, evaluation, and final prediction generation.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
SCREENSHOT_DIR = Path("screenshots")
OUTPUT_DIR.mkdir(exist_ok=True)
SCREENSHOT_DIR.mkdir(exist_ok=True)


## 1. Load the Dataset

In [ ]:
train = pd.read_csv(DATA_DIR / "house_prices_train.csv")
test = pd.read_csv(DATA_DIR / "house_prices_test.csv")

print("Training shape:", train.shape)
print("Testing shape:", test.shape)
train.head()


## 2. Data Quality Checks

In [ ]:
print("Data types:")
display(train.dtypes)

print("\nMissing values:")
display(train.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", train.duplicated().sum())


In [ ]:
display(train.describe(include="all").T.head(20))


## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(train["SalePrice"], bins=30)
plt.title("House Price Distribution")
plt.xlabel("Sale Price")
plt.ylabel("Number of Houses")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(train["GrLivArea"], train["SalePrice"], alpha=0.35)
plt.title("Living Area vs Sale Price")
plt.xlabel("Above-Ground Living Area (sq ft)")
plt.ylabel("Sale Price")
plt.show()


In [ ]:
quality_summary = train.groupby("OverallQual")["SalePrice"].mean().reset_index()
display(quality_summary)

plt.figure(figsize=(8,5))
plt.plot(quality_summary["OverallQual"], quality_summary["SalePrice"], marker="o")
plt.title("Average Sale Price by Overall Quality")
plt.xlabel("Overall Quality Rating")
plt.ylabel("Average Sale Price")
plt.show()


## 4. Prepare Data for Machine Learning

In [ ]:
X = train.drop(columns=["SalePrice"])
y = train["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 5. Train Multiple Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42, n_estimators=180, learning_rate=0.05, max_depth=3
    ),
    "Random Forest": RandomForestRegressor(
        random_state=42, n_estimators=180, max_depth=14, n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_valid, pred),
        "RMSE": np.sqrt(mean_squared_error(y_valid, pred)),
        "R2_Score": r2_score(y_valid, pred)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("R2_Score", ascending=False)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


## 6. Select the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

best_pred = best_model.predict(X_valid)

mae = mean_absolute_error(y_valid, best_pred)
rmse = np.sqrt(mean_squared_error(y_valid, best_pred))
r2 = r2_score(y_valid, best_pred)

print("Best Model:", best_model_name)
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")


## 7. Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_valid, best_pred, alpha=0.4)
mn, mx = min(y_valid.min(), best_pred.min()), max(y_valid.max(), best_pred.max())
plt.plot([mn, mx], [mn, mx])
plt.title(f"Actual vs Predicted Prices — {best_model_name}")
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.show()


## 8. Generate Final Predictions

In [ ]:
final_predictions = best_model.predict(test)
prediction_output = test[["HouseID"]].copy()
prediction_output["PredictedSalePrice"] = np.round(final_predictions, 0).astype(int)

prediction_output.to_csv(OUTPUT_DIR / "house_price_predictions.csv", index=False)

display(prediction_output.head(10))


## 9. Key Findings

1. Overall house quality is strongly associated with sale price.
2. Larger above-ground living areas generally correspond to higher sale prices.
3. Property type and neighborhood introduce additional price differences.
4. Missing numeric values were handled using median imputation.
5. Missing categorical values were handled using the most frequent category.
6. Categorical variables were converted into machine-readable one-hot encoded features.
7. Multiple regression models were compared using MAE, RMSE, and R².
8. The best-performing model on the validation set was selected for final predictions.

## Conclusion

The project demonstrates an end-to-end data analytics and machine-learning workflow for house price prediction. The solution can be extended with hyperparameter tuning, cross-validation, external housing datasets, and deployment as a web application.
